<center><font size=10>Project NLP - Healthcare</center></font>
<center><font size=6>Retrieval-Augmented Generation (RAG)</center></font>

![link text](https://d2ms8rpfqc4h24.cloudfront.net/leveraging_ai_powered_virtual_health_assistants_for_enhanced_patient_education_13dabc22ce.jpg)

## **Problem Definition**

The healthcare industry is rapidly evolving, and professionals face increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. Quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

## Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand information overload, apply AI techniques to streamline decision-making, analyze its impact on diagnostics and patient outcomes, evaluate its potential to standardize care practices, and create a functional prototype demonstrating its feasibility and effectiveness.

## Questions to Answer



*  What is the protocol for managing sepsis in a critical care unit?
* What are the common symptoms of appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?
* What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?
* What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?
* What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?



## **Installing and Importing the Necessary Libraries**

In [3]:
# Installation for GPU llama-cpp-python
!CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.2.28  --force-reinstall --upgrade --no-cache-dir -q 2>/dev/null
!pip uninstall -y numpy pandas chromadb
!pip install -q numpy==1.26.0 pandas==2.1.4 pypdf==4.0.1 langchain==0.1.1 langchain-community==0.0.13 chromadb==0.4.18 sentence-transformers==2.3.1 huggingface_hub==0.23.2 tiktoken

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.4/9.4 MB 129.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 305.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 284.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 363.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 257.4 MB/s eta 0:00:00
Found existing installation: numpy 2.3.4
Uninstalling numpy-2.3.4:
  Successfully uninstalled numpy-2.3.4
Found existing installation: pandas 2.2.2
Uninstalling pandas-2.2.2:
  Successfully uninstalled pandas-2.2.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.5/58.5 kB 3.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) 

In [1]:
import json
import tiktoken

import pandas as pd
import numpy as np

# Function to download the model from the Hugging Face model hub
from huggingface_hub import hf_hub_download

# Importing the Llama class from the llama_cpp module
from llama_cpp import Llama

from langchain.text_splitter import RecursiveCharacterTextSplitter

from langchain_community.document_loaders import PyPDFDirectoryLoader, PyPDFLoader
from langchain_community.embeddings.sentence_transformer import (
    SentenceTransformerEmbeddings
)
from langchain_community.vectorstores import Chroma

import chromadb

from google.colab import userdata, drive

In [2]:
import warnings
warnings.filterwarnings('ignore')

## **Downloading Model and Loading**

In [3]:
# Defining the Hugging Face repository and model version for Mistral-7B fine-tuned for instruction-following
model_name_or_path = 'TheBloke/Mistral-7B-Instruct-v0.2-GGUF'

# Specifying the file name for the quantized Mistral-7B model in GGUF format (Q6_K for optimal performance)
model_basename = 'mistral-7b-instruct-v0.2.Q6_K.gguf'

# Downloading the specified model file from Hugging Face Hub and store its local path
model_path = hf_hub_download(
    repo_id=model_name_or_path, #The Hugging Face repository containing the model
    filename=model_basename  # The specific model file to download (in GGUF format)
)
#The GGUF format is used because it provides memory-efficient storage and faster inference while maintaining compatibility across different hardware platforms.

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

In [4]:
llm = Llama(
    model_path=model_path, #Path to the GGUF model file
    n_ctx=2300, #Sets the context window to 2300 tokens (how much text the model can "see" at once)
    n_gpu_layers=38, #Loads 38 model layers onto GPU for faster inference (set to 0 for CPU-only)
    n_batch=512, #Number of tokens processed at once
    #verbose=False # Disable verbose output to avoid the fileno error
)

AVX = 1 | AVX_VNNI = 0 | AVX2 = 1 | AVX512 = 1 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 0 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


## **Model Performance check with Basic Parameters**

In [5]:
def response(query, max_tokens=512, temperature=0, top_p=0.95, top_k=50):
    # Sends the query prompt to the LLM with specified generation parameters
    model_output = llm(
        prompt=query, #The user's input question or prompt sent to the LLM
        max_tokens=max_tokens, #Maximum number of tokens to generate
        temperature=temperature, #Controls randomness
        top_p=top_p, #picks from top tokens that make up top_p of total probability
        top_k=top_k #considers only the top_k most likely tokens
    )
    # Extracting and returning only the text part of the response
    return model_output['choices'][0]['text'].strip()

In [6]:
print(response('What is the protocol for managing sepsis in a critical care unit?'))


Sepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:

1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.
2. ABCs: Ensure airway patency, adequate breathing, and circulatory support. Provide high-flow oxygen via a non-rebreather mask or endotracheal tube if necessary. Initiate intravenous fluids to maintain adequate blood pressure and organ perfusion.
3. Antibiotics: Administer broad-spectrum antibiotics based on the suspected source of infection and local microbiology data. Consider obtaining cultures before administering antibiotics if feasible.
4. Source control: Iden

In [7]:
print(response('What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'))

Llama.generate: prefix-match hit


Appendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:

1. Abdominal pain: The pain is typically located in the lower right quadrant of the abdomen and may start as a mild discomfort that gradually worsens over time. The pain may be constant or intermittent and may be aggravated by movement, deep breathing, or coughing.
2. Loss of appetite: People with appendicitis may lose their appetite due to abdominal pain or nausea.
3. Nausea and vomiting: Vomiting is a common symptom of appendicitis, especially in the later stages of the condition.
4. Fever: A fever of 100.4°F (38°C) or higher may be present in people with appendicitis.
5. Constipation or diarrhea: Some people with appendicitis experience constipation, while others have diarrhea.
6. Rebound tenderness: When the doctor presses on the abdome

In [8]:
print(response('What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'))


Llama.generate: prefix-match hit


Sudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, and eyelashes.

The exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications.

There are several treatments that have been shown to be effective in addressing sudden patchy hair loss:

1. Corticosteroids: These are anti-inflammatory drugs that can help reduce inflammation and suppress the immune system's attack on the hair follicles. They can be applied topically or taken orally, depending on the severity of the condition.
2. Minoxidil: This is a medication that has been shown to promote hair growth in some people with alopecia areata. It works by increasing blood 

In [9]:
print(response('What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'))

Llama.generate: prefix-match hit


A person who has sustained a physical injury to the brain, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for individuals with TBIs:

1. Emergency care: In case of a severe TBI, the first priority is to provide emergency care to ensure the person's airway is clear, breathing is stable, and circulation is adequate. This may involve intubation, oxygen therapy, and intravenous fluids.
2. Surgery: Depending on the location and severity of the injury, surgery may be necessary to remove hematomas (clots) or repair skull fractures.
3. Medications: Various medications may be prescribed to manage symptoms associated with TBIs, such as pain relievers for headaches, anti-seizure medications, and sedatives to help the person relax and rest.
4. Rehabilitation: Rehabilitation is an essential component of treatment for individuals with TBIs. This may include physical therapy

In [10]:
print(response('What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'))


Llama.generate: prefix-match hit


First and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:

1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.
2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.
3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement. Be sure not to apply too much pressure on the injury site.
4. Provide pain relief: Offer over-the-counter pain medication, such as acetaminophen or ibuprofen, to help manage pain.
5. Seek medical attention: If the fracture is severe or if you suspect that there may be other injuries, call for emergency medical assistance.

Once you've ensured the person's safety and stability, conside

### Obervation



*   High Medical Appropriateness: The model consistently generates answers that are medically accurate and appropriate for the stated conditions (brain injury, hair loss, sepsis, fractured leg), validating its factual reliability in this domain.

* Comprehensive Content: Responses are generally detailed and cover a broad spectrum of necessary information, demonstrating strong knowledge acquisition (e.g., listing multiple treatments or protocol steps).

* Excellent Structure and Readability: Answers are presented in a clear, organized format, frequently using numbered lists or distinct sections, which makes the complex medical information easy for a user to read and follow.

* Strong Contextual Relevance: The model remains focused on the query's core topic, generating precise and relevant information without drifting off-topic.

*   Generic Content Depth: While answers are appropriate, they often feel high-level or like general guidelines (e.g., standard protocol steps). They lack the nuance and situational depth that might be required for specific clinical scenarios.

*   Need for Contextual Specificity: The model defaults to useful but basic responses. The information provided is generally applicable rather than highly tailored, implying a need to fine-tune the prompt to demand greater contextual precision.

Lets leverage prompt engineering to enhance the quality of our responses.




## **Q&A with Prompt Engineering to define parameters**

Lets change the LLM parameter and check the effectiveness of the response

* temp (Temperature): A scalar value that controls the randomness (or "creativity") of the model's output by scaling the probability distribution of the next token.
* top_p (Nucleus Sampling): A dynamic sampling filter that restricts the choice of the next token to the smallest set of highest-probability tokens whose cumulative probability exceeds the threshold $p$.
* top_k: A static sampling filter that restricts the model's choice to only the $K$ most likely tokens at each step.
* max_tokens: A hard limit that defines the maximum number of tokens the model is allowed to generate in its response.

### Response function

In [11]:
def generate_llama_response(query, temperature, top_p, top_k, max_tokens, instruction="Answer the following question clearly and concisely."):

    # Create system message with instructions for the model
    system_message = f"[INST]<<SYS>>\n{instruction}\n<</SYS>>[/INST]"

    # Construct the final prompt using the user's query and system message
    prompt = f"{system_message} {query}"

    # Generate a response using the LLaMA model
    response = llm(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        top_p=top_p,
        repeat_penalty=1.1,
        top_k=top_k,
        stop=['</s>'],
        echo=False,
        seed=42,
    )

    # Return only the generated answer text
    return response["choices"][0]["text"].strip()

### Changing Temp Parameter (0.7 vs 0)

In [12]:
query= 'What is the protocol for managing sepsis in a critical care unit?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


The management of sepsis in a critical care unit involves the following steps:

1. Early recognition and diagnosis: Identify sepsis suspects based on clinical suspicion or use of early warning scores, and initiate sepsis assessment within one hour of identification.

2. Immediate intervention: Begin fluid resuscitation with 30 mL/kg crystalloid for hypotension or lactate level >2 mmol/L. Consider administering broad-spectrum antibiotics as soon as possible based on culture and sensitivity results, if not already initiated.

3. Source control: Address the infection source, such as removing catheters, draining abscesses, or performing surgery.

4. Adjusting fluid management: Assess the need for additional fluids based on hemodynamic response to initial resuscitation and ongoing fluid losses. Monitor urine output and central venous pressure.

5. Vasopressor support: Use vasopressors if necessary to maintain mean arterial pressure ≥65 mmHg, while minimizing the risk of tissue ischemia.

6.

#### temperature = 0

In [13]:
query= 'What is the protocol for managing sepsis in a critical care unit?'
temperature = 0.0
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


The management of sepsis in a critical care unit involves early recognition, prompt initiation of antibiotics, fluid resuscitation, and supportive measures. The Sequential [Sepsis-related] Organ Failure Assessment (SOFA) score can be used to identify patients at risk for developing sepsis or worsening sepsis. Once sepsis is suspected, blood cultures should be obtained and broad-spectrum antibiotics administered based on the patient's clinical presentation and local microbiology data. Fluid resuscitation with crystalloids or colloids may be necessary to maintain adequate tissue perfusion. Vasopressors and/or mechanical ventilation may also be required for hemodynamic support. Corticosteroids, anticoagulation, and other adjunctive therapies may be considered based on individual patient needs and clinical trials. Close monitoring of vital signs, laboratory values, and organ function is essential to assess response to treatment and adjust therapy as needed.


#### Observation

Temp 0.7 (Superior): Generated comprehensive, nuanced clinical protocol with specific medical metrics, resulting in highly useful, human-like output due to greater token exploration.

Temp 0.0 (Inferior): Produced incomplete, truncated, generic content, exhibiting a stiff, deterministic style that lacked critical detail for decision-making.

### Changing top_p Parameter (0.95 vs 0.85)

In [14]:
query= 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


Appendicitis is an inflammatory condition of the appendix, a small pouch that extends from the large intestine. The most common symptoms are:
1. Sudden pain in the lower right abdomen, sometimes starting as a mild pain that develops into sharp cramps over several hours.
2. Loss of appetite and feeling sick to your stomach (nausea).
3. Fever and chills.
4. Abdominal swelling.
5. Inability to pass gas or have a bowel movement.

Appendicitis cannot be cured via medicine alone, as the inflammation can quickly lead to the appendix rupturing, resulting in peritonitis - a serious infection of the abdominal cavity. The standard treatment for appendicitis is surgical removal of the appendix through a procedure called an appendectomy. This surgery can be performed either open or laparoscopically, depending on various factors including the severity of the condition and the patient's overall health.

Prompt diagnosis and timely surgical intervention are crucial for successful treatment, as complic

#### top_p=0.85

In [15]:
query= 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
temperature = 0.7
top_p = 0.85
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


Appendicitis is characterized by symptoms such as:
1. Sudden pain in the lower right abdomen, often starting around the belly button and migrating to the right.
2. Loss of appetite.
3. Nausea and vomiting.
4. Fever (usually over 101°F or 38.3°C).
5. Abdominal swelling and tenderness.
6. Constipation or diarrhea.
7. Inability to pass gas or have a bowel movement.

The condition cannot be cured with medicine alone, as the appendix cannot heal itself once it becomes inflamed or ruptures. The standard treatment for appendicitis is surgical removal of the appendix, which is known as an appendectomy. This procedure can typically be done through laparoscopic surgery, making the recovery process faster and less painful compared to traditional open surgeries. In some cases where the inflammation has spread beyond the appendix or if there are complications, a more extensive procedure like an open appendectomy may be required. It's essential to consult a healthcare professional for proper diagnos

#### Observation

* Answer 2 ($\text{top_p} = 0.85$): Superior—The narrower range created more focused and relevant content, minimizing irrelevant variation.
* Answer 1 ($\text{top_p} = 0.95$): Inferior—The wider range led to more diverse but general content and less precise output.
* Conclusion: A slightly lower $\text{top_p}$ improved quality by demanding greater concentration on the most probable tokens.

### Changing top_k Parameter (50 vs 80)

In [16]:
query= 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


There are several potential causes of sudden, patchy hair loss, including:

1. Alopecia Areata: This is an autoimmune condition that results in round patches of hair loss on the scalp or other parts of the body. Treatments for alopecia areata include topical corticosteroids, immunomodulators like minoxidil, and systemic medications like corticosteroids.
2. Traction Alopecia: This is a type of hair loss caused by pulling or tension on the hair, often seen in people who wear tight braids or ponytails. Treatment for traction alopecia involves avoiding hairstyles that pull on the hair and allowing the hair to grow back naturally.
3. Tinea Capitis: This is a fungal infection of the scalp that can cause patchy hair loss, often accompanied by scaling and redness. Treatment for tinea capitis involves antifungal medications, either topical or oral.
4. Nutritional Deficiencies: Certain nutritional deficiencies, such as iron deficiency, can lead to patchy hair loss. Treatment for nutritional defi

#### top_k=80

In [17]:
query= 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
temperature = 0.7
top_p = 0.95
top_k = 80
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


There are several potential causes of sudden, patchy hair loss, including:

1. Alopecia Areata: This is an autoimmune condition that results in round patches of hair loss on the scalp or other parts of the body. Treatments for alopecia areata include topical corticosteroids, immunomodulators like minoxidil, and systemic medications like corticosteroids.
2. Traction Alopecia: This is a type of hair loss caused by pulling or tension on the hair, often seen in people who wear tight braids or ponytails. Treatment for traction alopecia involves avoiding hairstyles that pull on the hair and allowing the hair to grow back naturally.
3. Tinea Capitis: This is a fungal infection of the scalp that can cause patchy hair loss, often accompanied by scaling and redness. Treatment for tinea capitis involves antifungal medications, either topical or oral.
4. Nutritional Deficiencies: Certain nutritional deficiencies, such as iron deficiency, can lead to patchy hair loss. Treatment for nutritional defi

#### Observation

The comparison shows how increasing the $\text{top_k}$ value can influence the depth and specificity of the response.

* Answer 2 ($\text{top_k} = 80$): Superior. The wider $K$ allowed the model to access a broader vocabulary, resulting in deeper clinical detail and more actionable insights regarding specific treatments (Alopecia Areata, Traction Alopecia).
* Answer 1 ($\text{top_k} = 50$): Inferior. The narrower $K$ restricted the model, yielding a more general overview that listed causes and only briefly mentioned treatments.

* Conclusion: For this medical project, a higher $\text{top_k}$ (80) was more effective, providing the necessary level of clinical depth required for actionable results.

### Changing max_tokens Parameter (1024 vs 512)

In [18]:
query= 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


The treatment for a brain injury depends on the severity and location of the injury. Here are some common interventions:

1. Acute care: Immediate care may include surgery to remove hematomas or other obstructions, controlling swelling and preventing further damage through medications such as corticosteroids or mannitol, and managing increased intracranial pressure through intubation and mechanical ventilation.

2. Rehabilitation: Physical therapy, occupational therapy, speech-language therapy, and cognitive rehabilitation can help improve function, prevent complications, and promote independence following a brain injury.

3. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or depression.

4. Supportive care: This includes ensuring adequate nutrition, hydration, and maintaining a clean environment to prevent infections.

5. Assistive devices: Devices like wheelchairs, walkers, or communication aids can help individuals r

#### max_token=512

In [19]:
query= 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 512
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


The treatment for a brain injury depends on the severity and location of the injury. Here are some common interventions:

1. Acute care: Immediate care may include surgery to remove hematomas or other obstructions, controlling swelling and preventing further damage through medications such as corticosteroids or mannitol, and managing increased intracranial pressure through intubation and mechanical ventilation.

2. Rehabilitation: Physical therapy, occupational therapy, speech-language therapy, and cognitive rehabilitation can help improve function, prevent complications, and promote independence following a brain injury.

3. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as seizures, pain, or depression.

4. Supportive care: This includes ensuring adequate nutrition, hydration, and maintaining a clean environment to prevent infections.

5. Assistive devices: Devices like wheelchairs, walkers, or communication aids can help individuals r

#### Observation

The comparison clearly shows that increasing the $\text{Max_tokens}$ parameter directly translates to a more comprehensive and well-rounded response.

* Answer 1 ($\text{Max_tokens} = 1024$): Superior. Provided a comprehensive and detailed overview of brain injury treatment, including specific interventions and covering a wider range of recovery aspects (physical, cognitive, and emotional support).

* Answer 2 ($\text{Max_tokens} = 512$): Inferior. Was more concise, focusing only on key areas (initial care, medications, rehabilitation). While medically accurate, the answer was less detailed and omitted broader aspects of recovery.

* Conclusion: For complex topics like medical protocols, the larger $\text{Max_tokens}$ (1024) is preferred as it allows the model to fully elaborate and provide a more holistic and detailed answer.

### Basic vs top parameters

In [20]:
query= 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
temperature = 0.7
top_p = 0.95
top_k = 50
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


1. Assess the severity of the injury: If the fracture is open (compound) or the person is experiencing severe pain, swelling, or inability to move, call emergency services immediately for medical attention.
2. Keep the area immobile: Use a splint, sling, or brace to keep the leg as still as possible to prevent further damage and promote healing.
3. Apply ice: Apply an ice pack to the injured area for 15-20 minutes at a time every few hours to reduce swelling and pain. Do not apply ice directly to the skin; use a protective layer such as a towel or cloth.
4. Elevate the leg: Keep the fractured leg raised above heart level to minimize swelling and help reduce pain.
5. Pain management: Over-the-counter pain relievers like acetaminophen or ibuprofen can be taken as directed to manage pain. If prescription pain medication is needed, follow your healthcare provider's instructions carefully.
6. Monitor for signs of infection: Watch for signs of infection such as redness, warmth, increased swe

In [21]:
query= 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
temperature = 0.7
top_p = 0.85
top_k = 80
max_tokens = 1024
print(generate_llama_response(query, temperature, top_p, top_k, max_tokens))

Llama.generate: prefix-match hit


1. Assess the severity of the injury: If the fracture is open (compound), keep the injured area clean and covered to prevent infection. Do not attempt to realign the bone or apply pressure to the affected limb. Seek immediate medical attention.

2. Immobilize the leg: Use a splint, sling, or brace to support the injured leg and prevent further movement. This will help reduce pain and prevent complications such as swelling and muscle damage.

3. Manage pain: Over-the-counter pain medication can be used to manage discomfort. If the pain is severe, consult a healthcare professional for prescription pain relief.

4. Monitor vital signs: Keep an eye on the person's heart rate, breathing rate, and temperature. These indicators can provide valuable information about their overall condition and help identify potential complications.

5. Provide fluids: Encourage the person to drink plenty of water or other clear fluids to maintain hydration. Dehydration can slow down recovery and worsen sympto

#### Observation

* Modified Parameters (Answer 2) were Superior: Produced a holistic and structured response that covered both immediate care and long-term recovery (e.g., nutrition, physical therapy).

* Default Parameters (Answer 1) were Inferior: Focused too narrowly on immediate emergency steps (e.g., severity assessment), lacking a comprehensive follow-up plan.

* Conclusion: Modification successfully shifted the model from a simple emergency protocol to a complete, structured guide covering the full recovery process.

## **Loading the data**

In [5]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
pdf_file = "/content/drive/My Drive/Colab Notebooks/NLP_Project/medical_diagnosis_manual.pdf"

In [7]:
pdf_loader = PyPDFLoader(pdf_file)

In [8]:
merck = pdf_loader.load()

In [9]:
from langchain.schema import Document # Import the Document class

phrase_to_remove = "This file is meant for personal use by rajasekar.t7@gmail.com only."
cleaned_docs = []

for doc in merck:
    # Remove the specific contamination phrase and clean up whitespace
    cleaned_content = doc.page_content.replace(phrase_to_remove, "").strip()

    # Create a new Document object with the cleaned content and original metadata
    cleaned_doc = Document(page_content=cleaned_content, metadata=doc.metadata)
    cleaned_docs.append(cleaned_doc)

merck=cleaned_docs

In [10]:
len(merck)

4114

## **Retrieval Augmented Generation (RAG) Process**

### **Implementing RAG**

#### Chunking documents

In [11]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap=16,
    is_separator_regex=False,
)

The given code initializes a **RecursiveCharacterTextSplitter** to split the text into manageable chunks for embedding and retrieval. Here's a breakdown:

- `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)`: Uses **TikToken encoding** to properly handle token-based splitting.
- `encoding_name='cl100k_base'`: Specifies the **TikToken encoding** (used by OpenAI models like GPT-4 and GPT-3.5).
- `chunk_size=512`: Each text chunk will have a maximum of **512 tokens**.
- `chunk_overlap=16`: Ensures **overlapping** of 16 tokens between consecutive chunks to preserve context.

This approach ensures that text is split **intelligently** while maintaining **semantic meaning** for better retrieval and embeddings.

In [12]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [13]:
len(document_chunks)

8546

Let's take a look at consecutive chunks from the document.

In [14]:
i = 5
print(document_chunks[i])

page_content='719\nChapter 68. Retinal Disorders  \n  .................................................................................................................................................\n731\nChapter 69. Optic Nerve Disorders  \n  ......................................................................................................................................\n737\nChapter 70. Orbital Diseases  \n  ..................................................................................................................................................\n742\n7 - Dermatologic Disorders  \n  ....................................................................................................................................................\n742\nChapter 71. Approach to the Dermatologic Patient  \n  .......................................................................................................\n755\nChapter 72. Principles of Topical Dermatologic Therapy  \n  ................

In [15]:
print(document_chunks[i+1])

page_content='EB3R42OHCV\nThis file is meant for personal use by rajasekar.t7@gmail.com only.\nSharing or publishing the contents in part or full is liable for legal action.' metadata={'source': '/content/drive/My Drive/Colab Notebooks/NLP_Project/medical_diagnosis_manual.pdf', 'page': 3}


As we can see there is some overlap between the chunks. This improves the coherence and relevance of retrieved results, as the model can better understand the relationship between adjacent parts of the document. It also helps in maintaining the flow of ideas and ensuring that critical context is available when generating answers, leading to more accurate and contextually consistent outputs.


#### Choosing an embedding model

A good general-purpose embedding model is [`gte-large`](https://huggingface.co/thenlper/gte-large). The main reason for choosing this model is because of its embedding vector size, which is 512, the same as our token size in chunking.

In [16]:
embedding_model = SentenceTransformerEmbeddings(model_name='thenlper/gte-large')

The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.


0it [00:00, ?it/s]

modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [17]:
# Generating embedding for the first document chunk
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
# Generating embedding for the second document chunk
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

In [18]:
#Checking if both are of the same size
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

Now that we have chunked the raw input, **we can present these chunks to an embedding model and then store the generated embeddings into a vector database.**
  - We generate a vector for each chunk and save this chunk along with the vector representation in a specialized database.

#### Creating a Vector Database

While embeddings solve for retrieval of appropriate context given a query, a more efficient application of vectorization is to transform raw data into smaller chunks before feeding it to an embedding model. There are two important reasons why this is needed:

- Embedding models are themselves Transformer models and hence have input length constraints. Any text that is longer than the maximum input length allowed by the embedding model is usually truncated.
- Slicing the data into chunks facilitates fine-grained control of the specific information that can be injected as context. This helps the model focus only on the information most relevant to the query.

In [19]:
import os
out_dir = 'merck_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

In [20]:
vectorstore = Chroma.from_documents( #creating a Chroma vector store from a set of document chunks.
    document_chunks, #creating a list of text chunks that will be converted into embeddings..
    embedding_model, #model responsible for embedding the document chunks into vector representations
    persist_directory=out_dir #name of the collection in the Chroma database
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [21]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [22]:
print(vectorstore.embeddings)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
) model_name='thenlper/gte-large' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False


In [23]:
print(vectorstore.similarity_search("Alopecia Areata ",k=3))

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given


[Document(page_content='infection or emotional stress. It occasionally coexists with autoimmune vitiligo or thyroiditis.\nDiagnosis\n• Examination\nDiagnosis is by inspection. Alopecia areata typically manifests as discrete circular patches of hair loss\ncharacterized by short broken hairs at the margins, which resemble exclamation points. Nails are\nsometimes pitted or display trachyonychia, a roughness of the nail also seen in lichen planus. Differential\ndiagnosis includes tinea capitis, trichotillomania, discoid lupus, and secondary syphilis. Measures of\nthyroid-stimulating hormone, vitamin B\n12\n, and autoantibodies are indicated only when coexisting disease\nis suspected.\nTreatment\n• Corticosteroids\n• Sometimes topical anthralin, minoxidil or both\nTreatment is with corticosteroids. Triamcinolone acetonide suspension (in doses not to exceed 0.1 mL per\ninjection site, eg, 10 mg/mL concentration to deliver 1 mg) can be injected intradermally if the lesions are\nsmall. Potent 

#### Retrieval

The user input is converted to a vector representation using the same model that was used for the context chunks. Then a similarity search is executed against the vectorized chunks in the vector database. Top-$k$ chunks from the search results in this step are then stuffed into the prompt as the context and the LLM is instructed to answer the user query using only the context.

We will now create a retriever that can query an input text and retrieve the top$-k$ documents that are most relevant from the vector store.

- Under the hood, a similarity score is computed between the embedded query and all the chunks in the database
- The top $k$ chunks with the highest similarity scores are then returned.

In [24]:
retriever = vectorstore.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 3} #Retrieving the top 3 most similar documents for a given query.
)

In [25]:
user_input = 'What are the symptoms of migraine?'
rel_docs = retriever.get_relevant_documents(user_input)
print(rel_docs)

[Document(page_content='are less common than visual auras. Some patients have an aura with little or no headache.\nHeadache varies from moderate to severe, and attacks last from 4 hours to several days, typically\nresolving with sleep. The pain is often unilateral but may be bilateral, most often in a frontotemporal\ndistribution, and is typically described as pulsating or throbbing.\nMigraine is more than a headache. Associated symptoms such as nausea (and occasionally vomiting),\nphotophobia, sonophobia, and osmophobia are prominent. Patients report difficulty concentrating during\nattacks. Routine physical activity usually aggravates migraine headache; this effect, plus the photophobia\nand sonophobia, encourages most patients to lie in a dark, quiet room during attacks. Severe attacks canThe Merck Manual of Diagnosis & Therapy, 19th Edition\nChapter 178. Headache\n1886rajasekar.t7@gmail.com\nEB3R42OHCV\nThis file is meant for personal use by rajasekar.t7@gmail.com only.\nSharing or

### **System and User Prompt Template**

In [26]:
# System message instructing the LLM to only answer using Merck Manual 19th Edition
qna_system_message = """
You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.
"""

In [27]:
# Template for formatting the user's input with context from the Merck Manual, 19th Edition and the actual medical question.
qna_user_message_template = """
###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
{context}

###Question
{question}
"""

#### Response Function

In [28]:
def generate_rag_response(user_input,ret,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = ret.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

### **Q&A with RAG**

##### Basic Parameter

In [29]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
print(generate_rag_response(user_input1,retriever))

###Answer
The protocol involves administering broad-spectrum antibiotics such as gentamicin or tobramycin, along with a third-generation cephalosporin, ceftazidime, or a fluoroquinolone. Vancomycin should be added if resistant staphylococci or enterococci are suspected. If there is an abdominal source, a drug effective against anaerobes should be included. Culture and sensitivity results should be used to adjust the antibiotic regimen accordingly. Antibiotics should be continued for at


##### Removing chunk overlap

In [30]:
# Initializing a RecursiveCharacterTextSplitter to split the text into manageable chunks for embedding and retrieval
text_splitter1 = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=512,
    chunk_overlap= 0,
    is_separator_regex=False,
)

In [31]:
#loading the PDF document, extracting its text, and splitting it into smaller chunks
document_chunks1 = pdf_loader.load_and_split(text_splitter1)

In [32]:
len(document_chunks1)

8522

In [33]:
# Generating embedding for the first document chunk
embedding_1 = embedding_model.embed_query(document_chunks1[0].page_content)
# Generating embedding for the second document chunk
embedding_2 = embedding_model.embed_query(document_chunks1[1].page_content)

In [34]:
vectorstore1 = Chroma.from_documents( #creating a Chroma vector store from a set of document chunks.
    document_chunks1, #creating a list of text chunks that will be converted into embeddings..
    embedding_model, #model responsible for embedding the document chunks into vector representations
    persist_directory=out_dir #name of the collection in the Chroma database
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [35]:
 vectorstore1 = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


In [36]:
print(vectorstore1.embeddings)

client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False}) with Transformer model: BertModel 
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False})
  (2): Normalize()
) model_name='thenlper/gte-large' cache_folder=None model_kwargs={} encode_kwargs={} multi_process=False


In [37]:
retriever1 = vectorstore1.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 3} #Retrieving the top 3 most similar documents for a given query.
)

In [38]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
print(generate_rag_response(user_input1,retriever1))

Llama.generate: prefix-match hit


Based on the context provided, the following are the steps for managing sepsis in a critical care unit:
1. Obtain specimens of blood, body fluids, and wound sites for Gram stain and culture before administering parenteral antibiotics.
2. Start very prompt empiric therapy immediately after suspecting sepsis.
3. Select an antibiotic regimen based on the suspected source, clinical setting, knowledge or suspicion of causative organisms and sensitivity patterns common to that specific inpatient unit, and previous culture results.
4. Consider adding vancomycin if


##### Observation

Removing chunk overlap did not change the answers in this case, probably because the relavant information for the question was already contained within a single chunk.

Removing chunk overlap might affect the answer if the question required information that spanned across multiple chunks from different parts of the text.

##### Basic Parameter

In [39]:
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
print(generate_rag_response(user_input2,retriever))

Llama.generate: prefix-match hit


The common symptoms for appendicitis include abdominal pain, anorexia (loss of appetite), and abdominal tenderness. Appendicitis cannot be cured via medicine alone as it requires surgical removal of the inflamed appendix. The treatment is called appendectomy, which can be performed either open or laparoscopically. Antibiotics are also administered before the surgery to prevent infection and reduce the risk of complications. If the appendix has already ruptured, antibiotics should be continued until the patient's temperature and white blood cell count have normalized. In some


##### Fine Tuning - Retreiver Parameters

In [40]:
retriever2 = vectorstore.as_retriever( #Converting the Chroma vector store into a retriever for querying.
    search_type='similarity', #Specifying that retrieval is based on cosine similarity
    search_kwargs={'k': 2} #Retrieving the top 2 most similar documents for a given query.
)

In [41]:
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
print(generate_rag_response(user_input2,retriever2))

Llama.generate: prefix-match hit


The common symptoms for appendicitis include abdominal pain, anorexia (loss of appetite), and abdominal tenderness. Appendicitis cannot be cured via medicine alone; the standard treatment is surgical removal of the appendix through open or laparoscopic appendectomy. This procedure should be preceded by IV antibiotics to prevent infection spread. If the appendix is perforated, antibiotics should be continued until the patient's temperature and WBC count have normalized. In cases where a large inflammatory mass is found involving the appendix, terminal ile


##### Observation

Answer 1, with k=3, is slightly more detailed and includes examples of cases (like when surgery is impossible).

Answer 2, with k=2, is slightly shorter and cuts off earlier, suggesting that retrieving more chunks (k=3) provided additional helpful context for a more complete answer.

##### Basic Parameter

In [42]:
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
print(generate_rag_response(user_input3, retriever))

Llama.generate: prefix-match hit


Based on the context provided, the possible causes of sudden patchy hair loss include alopecia areata, telogen effluvium, anagen effluvium, and other underlying disorders. The effective treatments or solutions for addressing this condition depend on the underlying cause. For alopecia areata, various treatment options exist including topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ul


##### Fine tuning temp=0.7

In [43]:
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
print(generate_rag_response(user_input3, retriever, temperature=0.7))

Llama.generate: prefix-match hit


The Merck Manual mentions alopecia areata as a cause of sudden patchy hair loss. It is an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers. The treatment options for alopecia areata include topical, intralesional, or systemic corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or psoralen plus ultraviolet A (PUVA). Hormonal modul


##### Observation

Answer 1, with temperature = 0, is more precise and structured.

Answer 2, with temperature = 0.7, introduces a bit more variability, with slight changes in wording, reflecting the increased randomness from the higher temperature.

##### Basic Parameter

In [44]:
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
print(generate_rag_response(user_input4,retriever))

Llama.generate: prefix-match hit


###Answer
The initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There is no specific treatment for a physical injury to brain tissue, but supportive


##### top_p=0.8

In [45]:
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
print(generate_rag_response(user_input4, retriever, top_p=0.8))

Llama.generate: prefix-match hit


###Answer
The initial treatment includes ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. There is no specific treatment for a physical injury to brain tissue, but supportive


##### Observation

Answer 1 with Top_p = 0.95 means the model will consider the top 95% probability mass of the next token and select from that pool. This makes the answer generated more focused, coherent responses because it prioritizes more likely words and phrases.

Answer 2 with Top_p = 0.8 means the model restricts the token selection to only the top 80% of the probability distribution, including less likely but more diverse options.

Thus, in Answer 1, the model focuses more on the main facts (like supportive care and rehab) without adding unnecessary or unrelated details. In Answer 2, the model explores a wider range of possibilities, leading to additional information about severe cases, surgery, and treatment phases.

##### Basic Parameter

In [46]:
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
print(generate_rag_response(user_input5,retriever))

Llama.generate: prefix-match hit


Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:
1. Antibiotics may be given as early as possible to prevent infection. The first dose may be given parenterally.
2. The wound should be immobilized using splints or a cast, depending on the severity of the injury and the length of the recovery period. Joints proximal and distal to the injury should also be immobilized.
3. The injured limb should be elevated above heart


##### top_k=25

In [47]:
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
print(generate_rag_response(user_input5,retriever, top_k=25))

Llama.generate: prefix-match hit


Based on the context provided, the following are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip:
1. Antibiotics may be given as early as possible to prevent infection. The first dose may be given parenterally.
2. The wound should be immobilized using splints or a cast, depending on the severity of the injury and the length of the recovery period. Joints proximal and distal to the injury should also be immobilized.
3. The injured limb should be elevated above heart


##### Observation

More or less it is the same output there is no much difference

## **Output Evaluation**

### Rating system

In [63]:
groundedness_rater_system_message = """

You will be presented a ###Question, ###Context used by the AI system and AI generated ###Answer.

Your task is to judge the extent to which the ###Answer is derived from ###Context.

Rate it 1 - if The ###Answer is not derived from the ###Context at all
Rate it 2 - if The ###Answer is derived from the ###Context only to a limited extent
Rate it 3 - if The ###Answer is derived from ###Context to a good extent
Rate it 4 - if The ###Answer is derived from ###Context mostly
Rate it 5 - if The ###Answer is is derived from ###Context completely

Please note: Make sure you give a single overall rating in the range of 1 to 5 along with an overall explanation.

"""

In [49]:
relevance_rater_system_message = """

You will be presented with a ###Question, the ###Context used by the AI system to generate a response, and the AI-generated ###Answer.

Your task is to judge the extent to which the ###Answer is relevant to the ###Question, considering whether it directly addresses the key aspects of the ###Question based on the provided ###Context.

Rate the relevance as follows:
- Rate 1 – The ###Answer is not relevant to the ###Question at all.
- Rate 2 – The ###Answer is only slightly relevant to the **###Question**, missing key aspects.
- Rate 3 – The ###Answer is moderately relevant, addressing some parts of the **###Question** but leaving out important details.
- Rate 4 – The ###Answer is mostly relevant, covering key aspects but with minor gaps.
- Rate 5 – The ###Answer is fully relevant, directly answering all important aspects of the **###Question** with appropriate details from the **###Context**.

Note: Provide a single overall rating in the range of 1 to 5, along with a brief explanation of why you assigned that score.
"""

In [50]:
user_message_template = """
###Question
{question}

###Context
{context}

###Answer
{answer}
"""

### Evaluation function

In [55]:
def generate_ground_relevance_response(user_input,ret,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = ret.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Print the prompt to inspect its length
    print("--- Generated Prompt ---")
    print(prompt)
    print("--- End of Generated Prompt ---")


    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        answer = response['choices'][0]['text'].strip()
    except Exception as e:
        answer = f'Sorry, I encountered the following error: \n {e}'

    # Combine user_prompt and system_message to create the prompt for groundedness and relevance raters
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Evaluation

#### Query1

In [58]:
user_input1 = 'What is the protocol for managing sepsis in a critical care unit?'
ground,rel = generate_ground_relevance_response(user_input1,retriever2,k=3,max_tokens=350)

print(ground,end="\n\n")
print(rel)

--- Generated Prompt ---

You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.


###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
16 - Critical Care Medicine
Chapter 222. Approach to the Critically Ill Patient
Introduction
Critical care medicine specializes in caring for the most seriously ill patients. These patients are best
treated in an ICU staffed by experienced personnel. Some hospitals maintain separate units for special
populations (eg, cardiac, surgical, neurologic, pediatric, or neonatal patients). ICUs have a high
nurse:patient ratio to provide the necessary high intensity 
of service, including treatment and monitoring
of ph

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 4 - The answer is derived from the context mostly.

Explanation: The context provides information about supportive care for ICU patients, which includes monitoring vital signs, providing adequate nutrition, preventing infection, maintaining fluid balance, and performing blood tests. The answer incorporates all of these elements in its protocol for managing sepsis in a critical care unit. Additionally, the context mentions point-of-care testing specifically for monitoring electrolyte levels and performing blood cultures, which is also included in the answer. However, the answer goes beyond the context by adding "administer antibiotics as soon as possible" and "provide emotional support and maintain patient dignity," which are not explicitly mentioned in the context but are related to critical care medicine in general.

 Rating: 5

Explanation: The AI-generated answer directly addresses all the key aspects of managing sepsis in a critical care unit as outlined in the question fr

#### Observation

Rating is 5 -- grounding and relevance means the answer is fully substantiated by the context and is maximally pertinent to the query.

#### Query2

In [59]:
user_input2 = 'What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?'
ground,rel = generate_ground_relevance_response(user_input2,retriever2,k=3,max_tokens=350)


print(ground,end="\n\n")
print(rel)

--- Generated Prompt ---

You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.


###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
Appendicitis is acute inflammation of the vermiform appendix, typically resulting in abdominal
pain, anorexia, and abdominal tenderness. Diagnosis is clinical, often supplemented by CT or
ultrasound. Treatment is surgical removal.
In the US, acute appendicitis is the most common cause of acute abdominal pain requiring surgery. Over
5% of the population develops appendicitis at some point. It most commonly occurs in the teens and 20s
but may occur at any age.
Other conditions affecting the appendix include car

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 5 - The AI-generated answer is completely derived from the context provided. The context mentions the symptoms, diagnosis, treatment options, and related conditions for appendicitis, as well as the information about hernias of the abdominal wall. The AI answer summarizes all these points accurately.

 Rating: 5

Explanation: The AI-generated answer directly addresses all important aspects of the question by providing a detailed explanation of the common symptoms for appendicitis, stating that it cannot be cured via medicine alone, and describing the standard surgical procedure for treating appendicitis. It also mentions the use of IV antibiotics before surgery and the possibility of resection of the entire mass if necessary. The answer is fully relevant to the question based on the provided context.


#### Observation

Rating is 5 -- grounding and relevance means the answer is fully substantiated by the context and is maximally pertinent to the query.

#### Query3

In [60]:
user_input3 = 'What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?'
ground,rel = generate_ground_relevance_response(user_input3,retriever2,k=3,max_tokens=350)



print(ground,end="\n\n")
print(rel)

--- Generated Prompt ---

You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.


###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
corticosteroids, retinoids, or immunosuppressants.
Hair loss due to chemotherapy is temporary and is best treated with a wig; when hair regrows, it may be
different in color and texture from the original hair. Hair loss due to telogen effluvium or anagen effluvium
is usually temporary as well and abates after the precipitating agent is eliminated.
Key Points
• Androgenetic alopecia (male-pattern and female-pattern hair loss) is the most common type of hair loss.
• Concomitant virilization in women or scarring

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 5 - The Answer is derived from the Context completely. The context mentions alopecia areata as a condition causing sudden patchy hair loss, and the Merck Manual mentioned in the context suggests corticosteroids, retinoids, or immunosuppressants as potential treatments for this condition. The Answer accurately reflects this information from the Context.

 Rating: 5

Explanation: The question asks about effective treatments or solutions for sudden patchy hair loss, specifically mentioning localized bald spots on the scalp. The context provided discusses alopecia areata, which is a condition characterized by sudden patchy hair loss with no obvious skin or systemic disorder. The answer directly addresses the question by stating that corticosteroids, retinoids, and immunosuppressants may be effective treatments for this condition as suggested in the context. Additionally, the answer explains that the cause behind alopecia areata is unclear but may involve genetic susceptibility and

#### Observation

Rating is 5 -- grounding and relevance means the answer is fully substantiated by the context and is maximally pertinent to the query.

#### Query4

In [61]:
user_input4 = 'What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?'
ground,rel = generate_ground_relevance_response(user_input4,retriever2,k=3,max_tokens=350)


print(ground,end="\n\n")
print(rel)

--- Generated Prompt ---

You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.


###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
Chapter 324. Traumatic Brain Injury
Introduction
Traumatic brain injury (TBI) is physical injury to brain tissue that temporarily or permanently
impairs brain function. Diagnosis is suspected clinically and confirmed by imaging (primarily
CT). Initial treatment consists of ensuring a reliable airway and maintaining adequate
ventilation, oxygenation, and blood pressure. Surgery is often needed in patients with more
severe injury to place monitors to track and treat intracranial pressure, decompress the brain i

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 5 - The AI generated answer is completely derived from the context. The context provides detailed information about the diagnosis, initial treatment, and subsequent care for a person with a traumatic brain injury (TBI). The AI answer accurately summarizes these key points.

 Rating: 5

Explanation: The AI-generated answer directly addresses the key aspects of the question by discussing the initial treatment for a person with a traumatic brain injury (TBI), which includes ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. It also mentions that surgery may be needed in more severe cases to treat intracranial pressure or remove hematomas. The answer also mentions the importance of preventing complications in the first few days after the injury and the need for rehabilitation subsequently. All of these points are directly related to the question and come from the provided context.


#### Observation

Rating is 5 -- grounding and relevance means the answer is fully substantiated by the context and is maximally pertinent to the query.

#### Query5

In [62]:
user_input5 = 'What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?'
ground,rel = generate_ground_relevance_response(user_input5,retriever2,k=3,max_tokens=350)


print(ground,end="\n\n")
print(rel)

--- Generated Prompt ---

You are a helpful assistant trained to answer questions based only on the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition.
Use the context provided to find accurate and reliable answers.
If the answer is not found in the context, reply with "I don't know".
Do not mention the context or the Merck Manual in your final answer.


###Context
The following excerpts are from the Merck Manual of Medical Diagnosis and Therapy, Nineteenth Edition:
• Possibly intraoral lacerations
• Some heavily contaminated wounds
If deemed necessary, antibiotics are given as early as possible; the first dose may be given parenterally.
Wounds are immobilized
 because excess movement of the affected area may interfere with healing.
Wounds near joints should be immobilized with splints. Bulky dressings are used to immobilize fingers and
hands. Wounds should be elevated, above heart level when feasible, for the first 48 h after suturing. A
sling may help maintain some deg

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Rating: 5 - The Answer is derived from the Context completely.

Explanation: The context provided in the text discusses various aspects of wound care, immobilization, and recovery for injuries such as lacerations and fractures. The question asks about the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, which is directly related to the information presented in the context. The answer summarizes the key points from the context regarding immobilization, antibiotic therapy, elevation, and patient care instructions for a person with a fractured leg. Therefore, the answer is derived completely from the context.

 Rating: 5

Explanation: The answer directly addresses all important aspects of the question based on the context provided. It covers necessary precautions such as early antibiotic therapy, immobilization using a cast or splints, elevation of the injured limb, and wound care instructions. It also mentions staying off feet for 

#### Observation

Rating is 5 -- grounding and relevance means the answer is fully substantiated by the context and is maximally pertinent to the query.

# **Conclusion**

**Business Benefits (Insights & Value)**

* High Trust & Reliability: RAG-based answers are context-specific and highly reliable, directly enhancing customer satisfaction and trust in medical applications.

* Superior User Experience (UX): The model provides personalized, relevant, and accurate responses, leading to a significantly improved user experience.

* Cost Efficiency: Automating medical Q&A dramatically reduces the necessity for human consultations, leading to substantial cost savings in telemedicine and healthcare support services.

**Strategic Recommendations (Future Action)**

* Mandatory Continuous Fine-tuning: The model requires ongoing updates and fine-tuning to reflect the latest medical guidelines and research (ensuring information remains current).

* Develop Specialized Models: Create dedicated models tailored to individual medical specialties (e.g., orthopedics, oncology) to maximize accuracy and relevance within specific domains.